# RAG Definition
Retrieval - Augmentation - Generation

LLMs are like a very smart student who have read the entire internet but have a "cutoff date". They don't know what happened yesterday, and they don't know what's inside your private files.

RAG (Retrieval-Augmented Generation) is the industry-standard way to fix this **without the massive cost of retraining the AI**. A RAG system retrieves relevant data to provide accurate responses.

| Features | RAG | Fine-Tuning |
|---|---|---|
|Updates| Instant: Just add a new file to the DB. | Slow: requires time to retraining |
|Accuracy| Higher: Can cite specific resources | High: Can still halucinate facts |
|Cost| Low: just use of-the-shelf LLMs Models | High: requires high power computing to train model |
|Privacy| Secure: Data stays in your database. | Risky: data is baked in to the model |

### Important Terms:
- Retrieval: finding the most relevant information.  
- Augmented: adding that information to the prompt.  
- Generation: creating the final answer from the model.  
- Knowledge Base: the stored documents/data to search in.  
- Chunks: small pieces of a document.  
- Embedding: a numeric vector that represents text meaning.

### Big idea of RAG
1. Ingest data (documents, plain text, images — use OCR on images if needed) into your knowledge base.  
2. Split documents into chunks that fit the model's context window.  
3. Convert chunks to embeddings with an embedding model and store them in a vector database.  
4. At query time, retrieve relevant chunks via nearest-neighbor search in the vector DB.  
5. Augment the prompt with those passages.  
6. Use the LLM to generate the final answer conditioned on the augmented context.

# Codebase
Hands-on minimal implementation: ingestion → chunking → embedding → storage → retrieval → generation

In [2]:
# Configuration
# pip: numpy requests
import os
import numpy as np
from dotenv import load_dotenv

# Embedding
from huggingface_hub import InferenceClient

# LLM
from openrouter import OpenRouter

load_dotenv()
# Set these or use environment variables
EMBEDDING_API_KEY = os.getenv('HF_TOKEN')
LLM_API_KEY = os.getenv('OPENROUTER_API_KEY')

# Fallback embedding dim for deterministic local embeddings
EMBEDDING_DIM = 32

## Ingestion
Load documents into a list of dicts: {id, text, meta}.

In [3]:
# LOAD DOCUMENTS
# Load documents data from sample-documents folder
def load_docs(folder='sample-documents'):
    docs = []
    exts = {'.md', '.txt'}
    for root, _, files in os.walk(folder):
        for fn in files:
            if os.path.splitext(fn)[1].lower() in exts:
                path = os.path.join(root, fn)
                try:
                    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                        text = f.read().strip()
                    doc_id = os.path.relpath(path, folder).replace(os.sep, '_')
                    docs.append({'id': doc_id, 'text': text, 'meta': {'source': fn, 'path': path}})
                except Exception as e:
                    print(f"Error reading {path}: {e}")
    return docs

docs = load_docs()
len(docs)

2

In [4]:
# CHUNKING
# cutting the text into smaller pieces so it fits into the embedding model context window
def chunk_text(text, chunk_size=400, overlap=50):
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return chunks

# Quick test
chunk_text(docs[0]['text'], chunk_size=1000, overlap=50)[:2]

['---\ntitle: "Hometown Overview"\nauthor: "Your Name"\nsource: "personal"\ncreated: "2026-03-21"\ntags: [hometown, local-history, guide]\n---\n\n# My Hometown\n\nOverview\n--------\n\nMy hometown is a medium-sized city located in the temperate region of the country. It combines a long history with modern amenities and a friendly community. The town is known for its tree-lined streets, a historic downtown area, and a mix of industry, small businesses, and agriculture in the surrounding countryside.\n\nQuick facts\n- Population: ~75,000 (approx.)\n- Region: Central valley \n- Founded: 1800s (historic settlement)\n- Language(s): Primary local language and common secondary languages\n\nHistory\n-------\n\nThe town began as a small trading post in the 19th century and grew with the arrival of the railroad. Early industries included milling, agriculture, and later small-scale manufacturing. The historic district preserves several 19th- and early-20th-century buildings, including the old cou

In [5]:
# Embeddings via Hugging Face Inference API
HF_TOKEN = EMBEDDING_API_KEY
# preferred HF model for sentence embeddings
HF_EMBED_MODEL = 'sentence-transformers/all-mpnet-base-v2'

def get_embeddings(texts, batch_size=16):
    if HF_TOKEN:
        client = InferenceClient(api_key=HF_TOKEN)
        embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            resp = client.feature_extraction(batch, model=HF_EMBED_MODEL)
            if isinstance(resp, list) and len(resp) == len(batch) and isinstance(resp[0], list):
                embeddings.extend(resp)
            else:
                for t in batch:
                    r = client.feature_extraction(t, model=HF_EMBED_MODEL)
                    embeddings.append(r)
        normed = []
        for e in embeddings:
            arr = np.array(e, dtype=np.float32)
            n = np.linalg.norm(arr) + 1e-9
            normed.append((arr / n).tolist())
        return normed
    else:
        print("No HF token found")

# quick test
short_chunk = chunk_text(docs[0]['text'], chunk_size=300, overlap=50)

test_embeddings = get_embeddings(short_chunk, batch_size=2)

sample_vector = test_embeddings[0][:5]

formatted_sample = [f"{num:.4f}" for num in sample_vector]
print(f"🔍 Sample of Chunk 1 Vector: {formatted_sample} ...")

🔍 Sample of Chunk 1 Vector: ['-0.0572', '0.0281', '-0.0022', '0.0121', '0.0297'] ...


In [8]:
# Simple in-memory vector store
# items is a list of dicts: {id, embedding (list), text, meta}
class InMemoryVectorStore:
    def __init__(self): # creates an empty list to hold the vectors
        self.vectors = []  

    def upsert(self, items): # insert or update items in the vector store
        # maintain a simple id -> index map for faster updates
        if not hasattr(self, '_index'):
            self._index = {v['id']: i for i, v in enumerate(self.vectors)}
        for it in items:
            vid = it.get('id')
            if vid in self._index:
                # replace existing
                idx = self._index[vid]
                self.vectors[idx] = it
            else:
                self._index[vid] = len(self.vectors)
                self.vectors.append(it)

    def _cosine(self, a, b): # compute cosine similarity between two vectors
        a = np.array(a, dtype=np.float32)
        b = np.array(b, dtype=np.float32)
        denom = (np.linalg.norm(a) * np.linalg.norm(b)) + 1e-9
        return float(np.dot(a, b) / denom)

    def search_by_vector(self, query_vec, k=3): # return top-k most similar vectors to the query_vec
        scores = []
        for v in self.vectors:
            score = self._cosine(query_vec, v['embedding'])
            scores.append({'id': v['id'], 'score': score, 'text': v['text'], 'meta': v.get('meta')})
        scores.sort(key=lambda x: x['score'], reverse=True)
        return scores[:k]

    def __len__(self): # return number of vectors in the store
        return len(self.vectors)

# create store
store = InMemoryVectorStore()

## Retrieval
Nearest-neighbor search against the vector store (cosine similarity).

In [9]:
# Build prompt with retrieved contexts and simple citations
def build_prompt(query, contexts):
    system_prompt = ('Answer the question using only the provided sources. '
                     'If the answer is not in the sources, say "I dont know."')
    parts = []
    for i, c in enumerate(contexts, start=1):
        src = c.get('meta', {}).get('source', c.get('id'))
        parts.append(f"[Source {i}] {src} (score={c['score']:.3f}):\n{c['text']}\n")
    user_prompt = f"Question: {query}\n\nSources:\n" + '\n---\n'.join(parts) + '\n\nAnswer:'
    return system_prompt, user_prompt

# LLM Call with OpenRouter SDK
def generate_with_llm(prompt, max_tokens=256, temperature=0.0):
    # normalize prompt (support (system, user) tuple or plain string)
    if isinstance(prompt, tuple) and len(prompt) == 2:
        system_prompt, user_prompt = prompt
    else:
        system_prompt, user_prompt = None, prompt

    # Prefer OpenRouter SDK
    try:
        msgs = []
        if system_prompt:
            msgs.append({"role": "system", "content": system_prompt})
        msgs.append({"role": "user", "content": user_prompt})
        with OpenRouter(api_key=LLM_API_KEY) as client:
            resp = client.chat.send(model="z-ai/glm-4.5-air:free", messages=msgs)
        return resp.choices[0].message.content
    except Exception as e:
        print('OpenRouter SDK error, falling back to HTTP or local echo:', e)

## Augmentation
How retrieved contexts are added to the prompt and citation format.

In [10]:
# Demo: end-to-end minimal pipeline
def build_index(docs, chunk_size=800, overlap=100):
    items = []
    for doc in docs:
        chunks = chunk_text(doc['text'], chunk_size=chunk_size, overlap=overlap)
        for i, ch in enumerate(chunks):
            item = {
                'id': f"{doc['id']}_c{i}",
                'text': ch,
                'meta': {**doc.get('meta', {}), 'parent_id': doc['id'], 'chunk_index': i},
            }
            items.append(item)
    # batch embed
    texts = [it['text'] for it in items]
    embeddings = get_embeddings(texts)
    for it, emb in zip(items, embeddings):
        it['embedding'] = emb
    return items

items = build_index(docs)
store.upsert(items)
print('Indexed vectors:', len(store))

Indexed vectors: 16


## Generation
Uses the LLM to answer based on augmented prompt. Keep temperature low for factual answers.

In [11]:
# Query
query = 'Where is the region of the hometown?'
q_emb = get_embeddings([query])[0]
results = store.search_by_vector(q_emb, k=3)
print('\nRetrieved contexts:')
for r in results:
    print(f"- {r['id']} score={r['score']:.3f} meta={r['meta']}")

prompt = build_prompt(query, results)
print('\nPrompt preview:\n', prompt[:1000])
answer = generate_with_llm(prompt)
print('\nGenerated answer:\n', answer)


Retrieved contexts:
- hometown.md_c0 score=0.521 meta={'source': 'hometown.md', 'path': 'sample-documents\\hometown.md', 'parent_id': 'hometown.md', 'chunk_index': 0}
- hometown.md_c2 score=0.464 meta={'source': 'hometown.md', 'path': 'sample-documents\\hometown.md', 'parent_id': 'hometown.md', 'chunk_index': 2}
- hometown.md_c7 score=0.426 meta={'source': 'hometown.md', 'path': 'sample-documents\\hometown.md', 'parent_id': 'hometown.md', 'chunk_index': 7}

Prompt preview:
 ('Answer the question using only the provided sources. If the answer is not in the sources, say "I dont know."', 'Question: Where is the region of the hometown?\n\nSources:\n[Source 1] hometown.md (score=0.521):\n---\ntitle: "Hometown Overview"\nauthor: "Your Name"\nsource: "personal"\ncreated: "2026-03-21"\ntags: [hometown, local-history, guide]\n---\n\n# My Hometown\n\nOverview\n--------\n\nMy hometown is a medium-sized city located in the temperate region of the country. It combines a long history with modern am

In [ ]:
# Try ask anything in this mini RAG console app
# type 'quit' or 'exit' or 'q' to stop

def run_rag_console(store, top_k=3):
    """Mini RAG app: read user query, retrieve, and generate answer."""
    while True:
        q = input("Question (or 'quit' to exit): ").strip()
        if not q:
            continue
        if q.lower() in {'quit', 'exit', 'q'}:
            print("Exiting.")
            break
        try:
            emb_batch = get_embeddings([q])
            if not emb_batch:
                print("Failed to get embedding for the query.")
                continue
            q_emb = emb_batch[0]
            results = store.search_by_vector(q_emb, k=top_k)
            system_prompt, user_prompt = build_prompt(q, results)
            answer = generate_with_llm((system_prompt, user_prompt))
            print("\nRetrieved contexts:")
            for r in results:
                src = r.get('meta', {}).get('source', r['id'])
                print(f"- {src} (score={r['score']:.3f})")
            print("\nAnswer:\n", answer)
        except Exception as e:
            print("Error:", e)

# Start the mini app
run_rag_console(store, top_k=3)


Retrieved contexts:
- university.md (score=0.420)
- university.md (score=0.368)
- university.md (score=0.319)

Answer:
 Based on the provided sources, you studied sustainability in university. According to Source 2, sustainability was your focus area, which is described as "an interdisciplinary field bringing together ecology, social systems, policy, and design." Your studies included projects like Community Solar Feasibility Studies, Campus Waste Audits, and Green Infrastructure Mapping (Source 1), and you developed skills in GIS, lifecycle thinking, community engagement, and translating technical concepts for non-expert audiences (Source 3).

Retrieved contexts:
- hometown.md (score=0.244)
- university.md (score=0.208)
- university.md (score=0.193)

Answer:
 I dont know.
Exiting.
